# Exploratory Data Analysis (EDA): Mental Health & Social Graph

**Consulting Update - BigDataCompany**
To build a robust prototype for our client, this notebook leverages **Apache Spark** to analyze the foundational characteristics of the mental health text and social network datasets. By utilizing PySpark's distributed computing framework, we can efficiently profile text length distributions, uncover class imbalances in emotions, and explore graph topology at scale, ensuring our downstream machine learning models and graph queries are built on a solid understanding of the data's "4 V's" (Volume, Velocity, Variety, and Veracity).

In [0]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, length, size, split, count, isnan, when, desc

# Initialize Spark Session
spark = SparkSession.builder.appName("Advanced_EDA_MentalHealth").getOrCreate()

# Dynamically resolve the absolute path to the Datasets folder
current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, ".."))

mental_health_path = f"file:{repo_root}/Datasets/mental_health.csv"
twitter_graph_path = f"file:{repo_root}/Datasets/twitter_graph.csv"

mental_health_df = spark.read.csv(mental_health_path, header=True, inferSchema=True, multiLine=True, escape='"')
twitter_graph_df = spark.read.csv(twitter_graph_path, header=True, inferSchema=True)

### 1. Missing Data Assessment
Data veracity is critical. We first check for missing or null values across the textual dataset to determine if imputation or row-dropping is necessary before ML modeling.

In [0]:
from pyspark.sql.functions import lit

numeric_cols = {c for c, dtype in mental_health_df.dtypes if dtype in ("int", "bigint", "smallint", "tinyint", "float", "double")}

missing_counts = mental_health_df.select([
    count(when(col(c).isNull() | (isnan(c) if c in numeric_cols else lit(False)), c)).alias(c)
    for c in mental_health_df.columns
])
missing_counts.show()

### 2. Emotional Class Distribution (Visualized)
We aggregate the class distribution using Spark, then collect the small aggregated dataset to the driver node (via Pandas) for visualization. Identifying class imbalance here dictates whether we need to apply weighting or oversampling in our Spark MLlib pipeline.

In [0]:
emotion_counts_df = mental_health_df.groupBy("Emotion").count().orderBy(col("count").desc())

# Bring aggregated data to Pandas for plotting
emotion_pd = emotion_counts_df.toPandas()

plt.figure(figsize=(10, 5))
sns.barplot(data=emotion_pd, x="Emotion", y="count", palette="viridis")
plt.title("Distribution of Psychological Categories")
plt.xlabel("Category")
plt.ylabel("Volume of Texts")
plt.show()

### 3. Text Complexity: Word Count Analysis
Understanding how verbose users are when expressing different emotional states helps in tuning our NLP tokenization and embedding strategies.

In [0]:
# Calculate word counts using Spark SQL functions
mental_health_df = mental_health_df.withColumn("Word_Count", size(split(col("Text"), "\\s+")))

print("Overall Word Count Statistics:")
mental_health_df.select("Word_Count").summary("count", "mean", "min", "25%", "50%", "75%", "max").show()

# Average word count grouped by Emotion
print("Average Word Count by Emotion:")
mental_health_df.groupBy("Emotion").avg("Word_Count").orderBy(col("avg(Word_Count)").desc()).show()

### 4. Graph Topology: Node Centrality Proxies
By analyzing the social network's properties with PySpark DataFrames, we can identify "influencer" nodes and assess graph connectivity before shifting to GraphFrames for advanced motif finding and PageRank.

In [0]:
total_edges = twitter_graph_df.count()
unique_sources = twitter_graph_df.select("Source").distinct().count()
unique_destinations = twitter_graph_df.select("Destination").distinct().count()

print(f"Total Network Edges: {total_edges}")
print(f"Unique Source Nodes: {unique_sources}")
print(f"Unique Destination Nodes: {unique_destinations}")

# Find highest follower accounts acting as Source in the dataset
print("\nTop 5 Accounts with Most Followers (Source):")
twitter_graph_df.select("Source", "Source Followers", "Source Verified") \
    .dropDuplicates(["Source"]) \
    .orderBy(col("Source Followers").desc()) \
    .show(5)